# Workshop #4 — Executive Narrative Claims & Zero-Jargon Verification
**Course:** Data Analysis Programming (*Programación para Análisis de Datos*) — Semester 2026-II  
**Team 1:** Natural Hazard Monitoring Systems — Earthquake Activity Monitoring  
**Authors:** Juan Córdoba (`20211020097`), Angelo Ibáñez (`20212020007`), Mauricio Sánchez (`20212020045`)

---

## Objective
Every quantitative claim presented to **Dr. Elena Vance** in the Storytelling Dashboard (`dashboard/narrative.py`) must be **100% mathematically traceable** to the curated Gold summary dataset, adapt deterministically when filters change, include the permanent responsible-use disclaimer, and contain **zero** data-engineering jargon. This notebook independently recalculates every figure and asserts exact match inside `build_narrative()`.


In [1]:
import re
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dashboard.gold_loader import load_gold
from dashboard.narrative import build_narrative

story_df = load_gold("storytelling").df
brief = build_narrative(story_df)

for key in ("headline", "anomaly_summary", "energy_paradox", "regional_and_map_brief", "filter_scope_note", "disclaimer"):
    print(f"[{key.upper()}]\n  {brief[key]}\n")


[HEADLINE]
  Latest Monitored Day (2026-11-24 UTC): Global SAAI stands at -0.64 [Normal (< 1.0)] with 63 recorded earthquakes (peak M 5.3) across 46 observed days (4,076 total events in window).

[ANOMALY_SUMMARY]
  Anomalous Seismic Days Detected (2 days with SAAI ≥ 1.00): 2026-11-16 (Elevated, SAAI 1.53, 228 events, peak M 6.6); 2026-11-21 (Elevated, SAAI 1.08, 61 events, peak M 6.8).

[ENERGY_PARADOX]
  Volume vs. Energy Paradox: Earthquakes of M ≥ 5.5 represent only 2.31% of cataloged events (94 of 4,076), yet account for 80.77% of total radiated seismic energy. Meanwhile, shallow-focus events (0–70 km depth) comprise 89.01% of observations and 81.71% of released energy, concentrating potential ground-shaking exposure near the surface.

[REGIONAL_AND_MAP_BRIEF]
  4 of the top 15 active regions carry Elevated-or-higher alert status, led by MORO GULF, MINDANAO, PHILIPPINES (52 events, peak M 5.9), Sangay, Philippines (37 events, peak M 5.9), Yakutat, Alaska (33 events, peak M 5.7). O

## 1. Independent Mathematical Recalculation of Every Narrative Number


In [2]:
# 1. Temporal SAAI & Anomalous Days
ts = story_df[story_df["aggregation_type"] == "temporal_trend_saai"].sort_values("period_date")
latest = ts.iloc[-1]
anom = ts[ts["saai_score"] >= 1.0].sort_values("saai_score", ascending=False)

# 2. Volume vs. Energy Paradox & Shallow Share
sev = story_df[story_df["aggregation_type"] == "severity_distribution"]
dep = story_df[story_df["aggregation_type"] == "depth_regime_breakdown"]
tot_events = int(sev["event_count"].sum())
strong_plus = sev[sev["dimension_key"].isin(["Strong (5.5-6.4)", "Major/Great (>=6.5)"])]
m55_cnt = int(strong_plus["event_count"].sum())
m55_ev_pct = 100.0 * m55_cnt / tot_events
m55_en_pct = float(strong_plus["energy_share_pct"].sum())
shallow_row = dep[dep["dimension_key"] == "Shallow (0-70 km)"].iloc[0]

# 3. Regional Hotspots & Significant Map Epicenters
reg = story_df[story_df["aggregation_type"] == "top_regions_keywords"]
elev_reg_cnt = int((reg["alert_tier"] != "NORMAL").sum())
sig = story_df[story_df["aggregation_type"] == "recent_significant_events"].sort_values(
    ["mean_magnitude", "event_timestamp_utc"], ascending=[False, False]
)
top_sig = sig.iloc[0]
tsu_cnt = int(sig["tsunami_flag"].apply(lambda v: v is True or str(v).lower() == "true").sum())

expected_tokens = [
    ("Latest date", str(latest["period_date"]), brief["headline"]),
    ("Latest SAAI", f"{float(latest['saai_score']):+.2f}", brief["headline"]),
    ("Latest daily count", f"{int(latest['event_count']):,d}", brief["headline"]),
    ("Observed days count", f"{len(ts)} observed days", brief["headline"]),
    ("Total catalog events", f"{tot_events:,d}", brief["energy_paradox"]),
    ("M>=5.5 event count", f"{m55_cnt:,d} of {tot_events:,d}", brief["energy_paradox"]),
    ("M>=5.5 event %", f"{m55_ev_pct:.2f}%", brief["energy_paradox"]),
    ("M>=5.5 energy %", f"{m55_en_pct:.2f}%", brief["energy_paradox"]),
    ("Shallow event %", f"{float(shallow_row['percentage']):.2f}%", brief["energy_paradox"]),
    ("Shallow energy %", f"{float(shallow_row['energy_share_pct']):.2f}%", brief["energy_paradox"]),
    ("Anomalous day count", f"{len(anom)} days", brief["anomaly_summary"]),
    ("Elevated top-15 regions", f"{elev_reg_cnt} of the top 15", brief["regional_and_map_brief"]),
    ("Significant map count", f"{len(sig):,d} earthquakes", brief["regional_and_map_brief"]),
    ("Largest map magnitude", f"M {float(top_sig['mean_magnitude']):.1f}", brief["regional_and_map_brief"]),
    ("USGS tsunami bulletin count", f"{tsu_cnt} event", brief["regional_and_map_brief"]),
]

audit_rows = []
for claim_name, token, target_text in expected_tokens:
    matched = token in target_text
    audit_rows.append({"Claim": claim_name, "Expected_Token": token, "Verified_In_Narrative": matched})
    assert matched, f"Claim mismatch for {claim_name}: '{token}' not found in '{target_text}'"

display(pd.DataFrame(audit_rows))
print(f"[PASS] All {len(audit_rows)} quantitative claims verified with 100% exact fidelity!")


,Claim,Expected_Token,Verified_In_Narrative
0,Latest date,2026-11-24,True
1,Latest SAAI,-0.64,True
2,Latest daily count,63,True
3,Observed days count,46 observed days,True
4,Total catalog events,"4,076",True
5,M>=5.5 event count,"94 of 4,076",True
6,M>=5.5 event %,2.31%,True
7,M>=5.5 energy %,80.77%,True
8,Shallow event %,89.01%,True
9,Shallow energy %,81.71%,True


[PASS] All 15 quantitative claims verified with 100% exact fidelity!


## 2. Filter-Sensitivity Matrix & Zero-Jargon Audit


In [3]:
BANNED_JARGON = re.compile(
    r"\b(parquet|dataframe|dedup|deduplication|iqr|pipeline|schema|raw|silver|gold|spark|pyspark|bronze|medallion|etl|aggregation_type)\b",
    re.IGNORECASE,
)

scenarios = [
    ("Full Observation Window", None),
    ("Anomalous Days Only (Elevated+)", {"alert_tiers": ["ELEVATED", "HIGH", "CRITICAL"]}),
    ("Latest 3 Days + USGS Archive", {"start_date": "2026-11-22", "end_date": "2026-11-24", "agencies": ["usgs_fdsn"]}),
    ("Empty Selection Guardrail", {"alert_tiers": [], "agencies": []}),
]

scenario_rows = []
for name, flt in scenarios:
    res = build_narrative(story_df, filters=flt)
    jargon_hits = BANNED_JARGON.findall(res["full_text"])
    has_disclaimer = "does NOT predict" in res["disclaimer"]
    assert not jargon_hits and has_disclaimer
    scenario_rows.append({
        "Scenario": name,
        "Posture": res["posture_tier"],
        "Headline_Preview": res["headline"][:95] + ("..." if len(res["headline"]) > 95 else ""),
        "Banned_Jargon_Hits": len(jargon_hits),
        "Disclaimer_Present": has_disclaimer,
    })

display(pd.DataFrame(scenario_rows))


,Scenario,Posture,Headline_Preview,Banned_Jargon_Hits,Disclaimer_Present
0,Full Observation Window,NORMAL,Latest Monitored Day (2026-11-24 UTC): Global ...,0,True
1,Anomalous Days Only (Elevated+),ELEVATED,Latest Monitored Day (2026-11-21 UTC): Global ...,0,True
2,Latest 3 Days + USGS Archive,NORMAL,Latest Monitored Day (2026-11-24 UTC): Global ...,0,True
3,Empty Selection Guardrail,INFO,No daily seismic anomaly observations fall wit...,0,True
